# Demo 1 — Basic Agent + Tools

We'll spin up a Microsoft Agent Framework agent backed by **Microsoft Foundry** and give it two simple Python functions as tools — a (mocked) PubMed search and a gene lookup. The model decides when to call them.

**Prereqs**
- `pip install -r requirements.txt`
- A populated `.env` (see `.env.example`)
- `az login` so `AzureCliCredential` can authenticate

## 1. Load environment

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

PROJECT_ENDPOINT = os.environ["AZURE_AI_PROJECT_ENDPOINT"]
MODEL_DEPLOYMENT = os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"]

print("Foundry project:", PROJECT_ENDPOINT)
print("Model deployment:", MODEL_DEPLOYMENT)

## 2. Define some research tools

Plain Python functions with type hints and docstrings — the framework auto-generates the JSON schema the model sees. In production these would call real APIs (NCBI E-utilities, MyGene.info, an internal knowledge base, etc.); here we return small mocked payloads so the demo runs offline-friendly.

In [ ]:
from typing import Annotated
from pydantic import Field


def pubmed_search(
    query: Annotated[str, Field(description="Free-text PubMed query, e.g. 'TP53 pediatric ALL relapse'.")],
    max_results: Annotated[int, Field(description="Max number of results to return.")] = 3,
) -> list[dict]:
    """Return a list of recent (mocked) PubMed hits for the given query."""
    mocked = [
        {"pmid": "39812345", "year": 2025, "title": f"Single-cell atlas of {query} in pediatric cohorts"},
        {"pmid": "39798765", "year": 2024, "title": f"Functional CRISPR screen reveals dependencies in {query}"},
        {"pmid": "39654321", "year": 2024, "title": f"Long-term outcomes after targeted therapy in {query}"},
        {"pmid": "39512345", "year": 2023, "title": f"Genomic landscape of {query}: a multi-institutional study"},
    ]
    return mocked[:max_results]


def gene_lookup(
    symbol: Annotated[str, Field(description="HGNC gene symbol, e.g. 'TP53', 'IKZF1'.")],
) -> dict:
    """Return basic info (mocked) about a human gene by HGNC symbol."""
    catalog = {
        "TP53": {
            "chromosome": "17p13.1",
            "function": "Tumor suppressor; regulates cell-cycle arrest, apoptosis, DNA repair.",
            "disease_associations": ["Li-Fraumeni syndrome", "many cancers incl. pediatric ALL relapse"]
        },
        "IKZF1": {
            "chromosome": "7p12.2",
            "function": "Zinc-finger transcription factor; key regulator of lymphocyte differentiation.",
            "disease_associations": ["B-cell ALL (poor prognosis when deleted)"]
        },
        "MYCN": {
            "chromosome": "2p24.3",
            "function": "Proto-oncogene transcription factor.",
            "disease_associations": ["High-risk neuroblastoma (amplification)"]
        },
    }
    return catalog.get(symbol.upper(), {"error": f"No record for {symbol} in demo catalog."})

## 3. Create the agent against your Foundry project

In [ ]:
from agent_framework.foundry import FoundryChatClient
from azure.identity.aio import AzureCliCredential

credential = AzureCliCredential()
client = FoundryChatClient(
    project_endpoint=PROJECT_ENDPOINT,
    model=MODEL_DEPLOYMENT,
    credential=credential,
)

agent = client.as_agent(
    name="LiteratureTriageAgent",
    instructions=(
        "You are a research assistant helping pediatric oncology investigators triage "
        "literature and gene-level evidence. Use the available tools to fetch facts "
        "rather than relying on your training data. Cite PMIDs when you reference papers."
    ),
    tools=[pubmed_search, gene_lookup],
)

## 4. Ask it something that requires the tools

In [ ]:
result = await agent.run(
    "Give me a 3-sentence brief on TP53: what it does, what diseases it's tied to, "
    "and a couple of recent papers on TP53 in pediatric ALL relapse."
)
print(result)

## 5. Multi-turn follow-up

`AgentSession` keeps the conversation history so the agent remembers prior turns.

In [ ]:
from agent_framework import AgentSession

session = AgentSession()

r1 = await agent.run("I'm reviewing high-risk B-ALL cases this week.", session=session)
print("Agent:", r1)

r2 = await agent.run(
    "What gene is most associated with poor prognosis there, and pull a couple of recent papers on it?",
    session=session,
)
print("Agent:", r2)

## 6. Cleanup

In [ ]:
await credential.close()